# Car AI Project - Feature Engineering

#### Introduction

This notebook builds a model-ready feature set starting from the cleaned dataset produced in the EDA phase.

The objective is not to optimize a single algorithm, but to create a consistent and interpretable feature space that works across:

- Linear Regression (sensitive to missing values and scaling assumptions)
- Tree-based models (robust to missing values and non-linear relationships)

Feature engineering is guided by the insights from the EDA:

- strong non-linearity in price distribution
- clear separation between EV / ICE / Hybrid vehicles
- structural missingness in engine and battery features
- strong correlation between price and performance metrics

All transformations are designed to be:

- reproducible
- leakage-free
- model-agnostic where possible

## 1. Libraries

In [0]:
import pandas as pd
import numpy as np
import re
from datetime import datetime

In [0]:
# Import project configuration
import sys
import os

# Add parent directory to path to import config
sys.path.append("..")
from config import *

print("PROJECT CONFIGURATION LOADED")
print(f"\nBASE_PATH: {BASE_PATH}")
print(f"\nData Paths:")
print(f"   - RAW_DATA_PATH: {RAW_DATA_PATH}")
print(f"   - FEATURES_PATH: {FEATURES_PATH}")
print(f"   - PROCESSED_DATA_PATH: {PROCESSED_DATA_PATH}")
print(f"   - TRAIN_TEST_PATH: {TRAIN_TEST_PATH}")
print(f"   - METRICS_PATH: {METRICS_PATH}")
print(f"\nModel Path:")
print(f"   - MODEL_PATH: {MODEL_PATH}")
print(f"\nUnity Catalog:")
print(f"   - SOURCE_CSV_FILE: {SOURCE_CSV_FILE}")
print(f"   - RAW_CARS_TABLE: {RAW_CARS_TABLE}")
print(f"   - CLEANED_CARS_TABLE: {CLEANED_CARS_TABLE}")

PROJECT CONFIGURATION LOADED

BASE_PATH: /Workspace/Users/maria.petralia@gmail.com/CarAIProject

Data Paths:
   - RAW_DATA_PATH: /Workspace/Users/maria.petralia@gmail.com/CarAIProject/data/raw
   - FEATURES_PATH: /Workspace/Users/maria.petralia@gmail.com/CarAIProject/data/features
   - PROCESSED_DATA_PATH: /Workspace/Users/maria.petralia@gmail.com/CarAIProject/data/processed
   - TRAIN_TEST_PATH: /Workspace/Users/maria.petralia@gmail.com/CarAIProject/data/train_test
   - METRICS_PATH: /Workspace/Users/maria.petralia@gmail.com/CarAIProject/data/metrics

Model Path:
   - MODEL_PATH: /Workspace/Users/maria.petralia@gmail.com/CarAIProject/models

Unity Catalog:
   - SOURCE_CSV_FILE: /Volumes/workspace/caraiproject/caraiproject/Cars_Datasets_2025.csv
   - RAW_CARS_TABLE: workspace.caraiproject.raw_cars_data_gathered
   - CLEANED_CARS_TABLE: workspace.caraiproject.cleaned_cars_data


## 2. Load Dataset

### 2.1 Import Delta Table in a Pandas Dataframe

In [0]:
# Import the raw Delta Table into pandas DataFrame
# Define source table
source_table = "workspace.caraiproject.eda_cleaned_cars_data"

# Read Delta Table using Spark (required for Delta format)
spark_df = spark.table(source_table)
# Convert to pandas DataFrame for data manipulation
df = spark_df.toPandas()

print(f"\nDataset ready for Feature Engineering")


Dataset ready for Feature Engineering


In [0]:
# drop the cleaning_timestamp column
df = df.drop(columns=["cleaning_timestamp"])

## 3 Feature Engineering Strategy

Based on EDA findings, we define a unified feature engineering strategy:

1. **Powertrain Indicators** (EV / Hybrid structure)
2. **Fuel Macro-Category** (stable categorical simplification)
3. **Performance Composite Features** (reduce multicollinearity)
4. **Unified Engineered Dataset** (all models start from the same base)

**Philosophy:**

All models (Linear Regression, Tree-based) receive the **same feature set**.  
Model-specific adaptations (scaling, feature removal) are handled in the **preprocessing step**, not here.  
This ensures fair comparison and reproducible experimentation.

In [0]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1210 entries, 0 to 1209
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   make                    1210 non-null   object 
 1   model                   1210 non-null   object 
 2   engine_description      1210 non-null   object 
 3   engine_displacement_cc  1062 non-null   float64
 4   battery_capacity_kwh    104 non-null    float64
 5   acc_0_100_min           1205 non-null   float64
 6   acc_0_100_max           1205 non-null   float64
 7   acc_0_100_mean          1205 non-null   float64
 8   acc_0_100_is_range      1210 non-null   int64  
 9   acc_0_100_is_instant    1210 non-null   int64  
 10  is_commercial           1210 non-null   int64  
 11  horsepower_min          1210 non-null   float64
 12  horsepower_max          1210 non-null   float64
 13  horsepower_mean         1210 non-null   float64
 14  horsepower_is_range     1210 non-null   

In [0]:
df.duplicated().sum()


np.int64(1)

In [0]:
# drop duplicates
df = df.drop_duplicates()

In [0]:
df.shape

(1209, 20)

### 3.1 Powertrain Features

These features capture the structural differences between:

- Electric vehicles
- Hybrid vehicles
- Plug-in hybrids

In [0]:
# Electric vehicles
df['is_ev'] = df['fuel_type'].str.contains('Electric', case=False, na=False).astype(int)

In [0]:
# Hybrid vehicles (mild/full)
df['is_hybrid'] = df['fuel_type'].str.contains('Hybrid', case=False, na=False).astype(int)

In [0]:
# Plug-in Hybrid
df['is_phev'] = df['fuel_type'].str.contains('Plug', case=False, na=False).astype(int)

### 3.2 Fuel Macro Category

In [0]:
# Create a clean, complete macro-category for fuel types that:
# - contains no missing values
# - is stable and reproducible (based on 'fuel_types', which remains in df)
# - reflects the structure discovered during EDA
# - is suitable for both Linear Regression and tree-based models
def map_fuel_macro(x):
    if pd.isna(x):
        return 'Other'
    x = x.lower()
    if 'electric' in x:
        return 'Electric'
    if 'hybrid' in x:
        return 'Hybrid'
    if 'diesel' in x:
        return 'Diesel'
    if 'petrol' in x or 'gasoline' in x:
        return 'Petrol'
    return 'Other'

df['fuel_macro'] = df['fuel_type'].apply(map_fuel_macro)

In [0]:
# Feature validation
df[['fuel_type', 'fuel_macro', 'is_ev', 'is_hybrid', 'is_phev']].head()
df[['is_ev', 'is_hybrid', 'is_phev', 'fuel_macro']].isna().sum()

is_ev         0
is_hybrid     0
is_phev       0
fuel_macro    0
dtype: int64

**Note**: All engineered features are fully complete (no missing values), confirming that:

- feature logic is consistent with cleaned dataset
- no imputation leakage was introduced
- derived variables are safe for modeling

### 3.3 Segment & Performance-Based Features

**Rationale for the Segment Feature**

The `segment` feature assigns each vehicle to a mutually exclusive market category.  
Although a car may simultaneously exhibit multiple characteristics (e.g., a Bentley Continental GT is both luxury and sport-oriented), the segment must represent **the primary market identity** of the vehicle.

EDA results clearly show that **performance metrics are the strongest drivers of price**, more influential than brand alone:

- Horsepower has the highest correlation with log(price)
- Acceleration (0-100 km/h) shows a strong inverse relationship with price
- Top speed and torque form distinct performance clusters
- Luxury brands overlap heavily with sportscar and supercar clusters

This indicates that **performance dominates luxury** in determining market positioning.

For this reason, the segment is constructed using a **sequential hierarchy**, where the first matching rule defines the category:

1. Supercar (extreme performance)
2. Sportscar (high performance)
3. Luxury (comfort + prestige)
4. SUV Premium
5. SUV
6. Pickup / Commercial
7. Roadster
8. Electrified
9. Other

This hierarchy reflects real-world automotive pricing logic:

- Performance defines the primary identity of a vehicle  
- Segment (SUV, sedan, coupe, commercial) defines its functional role  
- Brand acts as a secondary price modifier, not the main classifier  

By enforcing a sequential rule-based approach, each vehicle receives **exactly one segment**, ensuring interpretability, stability, and full compatibility with machine learning models.


In [0]:
def map_segment(row):
    hp = row["horsepower_mean"]
    acc = row["acc_0_100_mean"]
    seats = row["seats"]
    top = row["top_speed_kmh"]
    make = row["make"]
    is_com = row["is_commercial"]
    is_ev = row["is_ev"]
    is_hybrid = row["is_hybrid"]

    # Supercar
    if hp > 600 or acc < 3.2:
        return "Supercar"

    # Sportscar
    if hp > 300 or acc < 5:
        return "Sportscar"

    # Luxury
    if make in ["Rolls-Royce", "Bentley", "Aston Martin"] and seats >= 4:
        return "Luxury"

    # SUV Premium
    if seats >= 5 and hp > 300 and top > 220:
        return "SUV Premium"

    # SUV
    if seats >= 5 and hp <= 300:
        return "SUV"

    # Pickup
    if is_com == 1:
        return "Pickup"

    # Roadster
    if seats == 2 and hp > 250:
        return "Roadster"

    # Electrified
    if is_ev == 1 or is_hybrid == 1:
        return "Electrified"

    return "Other"

df["segment"] = df.apply(map_segment, axis=1)

In [0]:
# Distribution table for the segment feature
segment_counts = df["segment"].value_counts().rename("count")
segment_percent = (df["segment"].value_counts(normalize=True) * 100).rename("percent")

segment_table = pd.concat([segment_counts, segment_percent], axis=1)
segment_table = segment_table.sort_values("count", ascending=False)

segment_table

,count,percent
segment,,
SUV,610,50.454921
Sportscar,360,29.776675
Supercar,119,9.842845
Other,100,8.271299
Electrified,7,0.578991
Pickup,7,0.578991
Roadster,6,0.496278


**Note**:  
This table shows the distribution of the `segment` feature.  
It confirms that the hierarchy produces a balanced and interpretable segmentation, with no excessively rare categories.  
The absence of a “Luxury” segment is expected: performance metrics (horsepower, acceleration) dominate brand effects, so high-end luxury brands with strong performance characteristics are absorbed into Sportscar or Supercar.

In [0]:
# Compact view: for each segment, list unique brands inside it
segment_to_brands = (
    df.groupby("segment")["make"]
      .apply(lambda x: ", ".join(sorted(x.unique())))
      .reset_index(name="brands")
      .sort_values("segment")
)
segment_to_brands

,segment,brands
0,Electrified,"Mitsubishi, Volkswagen"
1,Other,"Audi, BMW, Chevrolet, Ford, GMC, Hyundai, Jeep..."
2,Pickup,"Mazda, Mitsubishi"
3,Roadster,"Ford, Jaguar Land Rover, Mazda, Nissan, Volvo"
4,SUV,"Acura, Audi, BMW, Cadillac, Chevrolet, Ford, G..."
5,Sportscar,"Acura, Aston Martin, Audi, BMW, Bentley, Cadil..."
6,Supercar,"Acura, Aston Martin, Audi, BMW, Bugatti, Cadil..."


**Note**:  
This compact view lists, for each segment, the brands assigned to it.  
It helps validate that performance-oriented luxury brands (e.g., Ferrari, Lamborghini, Aston Martin, Bugatti, Porsche) are correctly classified as Sportscar or Supercar, while mainstream brands populate SUV and Other.  
Pure luxury brands (Rolls-Royce, Bentley) also appear in performance-based segments, reflecting the strong correlation between price and technical performance observed in the EDA.

### 3.4 Brand Tier Features

**Rationale for Brand Tier Features**

The EDA showed that brand alone is not the primary driver of price:  
performance metrics (horsepower, acceleration, top speed) have much stronger correlations with log(price).  
However, brand still contributes a secondary “premium uplift” that is useful for modeling.

For this reason, brand tiers are introduced as **binary indicators** (luxury, premium),  
complementing the performance-based `segment` feature without overriding it.

In [0]:
# Brand Tier Features
luxury_brands = ["Rolls-Royce", "Bentley", "Maybach"]

performance_luxury_brands = [
    "Ferrari", "Lamborghini", "Aston Martin", "Bugatti", "McLaren", "Maserati", "Porsche"
]

premium_brands = [
    "Mercedes-Benz", "BMW", "Audi", "Volvo", "Lexus", "Cadillac", "Lincoln",
    "Jaguar", "Land Rover", "Jaguar Land Rover", "Genesis", "Infiniti", "Acura"
]

df["is_luxury_brand"] = df["make"].isin(luxury_brands).astype(int)
df["is_performance_luxury_brand"] = df["make"].isin(performance_luxury_brands).astype(int)
df["is_premium_brand"] = df["make"].isin(premium_brands).astype(int)

In [0]:
brand_tier_check = (
    df.groupby("make")[["is_luxury_brand", "is_performance_luxury_brand", "is_premium_brand"]]
      .max()
      .sort_values(["is_luxury_brand", "is_performance_luxury_brand", "is_premium_brand"], ascending=False)
)

brand_tier_check

,is_luxury_brand,is_performance_luxury_brand,is_premium_brand
make,,,
Bentley,1,0,0
Rolls-Royce,1,0,0
Aston Martin,0,1,0
Bugatti,0,1,0
Ferrari,0,1,0
Lamborghini,0,1,0
Porsche,0,1,0
Acura,0,0,1
Audi,0,0,1


**Note**:  
The brand tier lists are defined as a general market classification, not limited to the brands present in this dataset.  
They include all major luxury, performance-luxury, and premium brands globally (Maybach, Maserati, McLaren, Lexus, Genesis), ensuring that the feature remains robust and scalable across different datasets.  
Brands not included in any tier default to “mainstream”.

### 3.5 Acceleration & Horsepower Classes

Acceleration & Horsepower Classes

Discrete classes are derived from `acc_0_100_mean` and `horsepower_mean` using fixed thresholds.  
These transformations reduce non-linearity and stabilize the effect of performance variables without introducing leakage.

##### Acceleration Class

In [0]:
# acceleration_class
def map_acc_class(acc):
    if acc < 3.2:
        return "Extreme"
    if acc < 5:
        return "Fast"
    if acc < 8:
        return "Medium"
    return "Slow"

df["acceleration_class"] = df["acc_0_100_mean"].apply(map_acc_class)

In [0]:
df["acceleration_class"].value_counts().rename("count")

acceleration_class
Slow       483
Medium     465
Fast       190
Extreme     71
Name: count, dtype: int64

##### Horsepower Class

In [0]:
def map_hp_class(hp):
    if hp > 600:
        return "Extreme"
    if hp > 300:
        return "High"
    if hp > 150:
        return "Medium"
    return "Low"

df["hp_class"] = df["horsepower_mean"].apply(map_hp_class)

In [0]:
df['hp_class'].value_counts().rename("count")


hp_class
Medium     441
High       371
Low        295
Extreme    102
Name: count, dtype: int64

### 3.7  Horsepower Binning (hp_bin) — Deferred to Preprocessing

The feature `hp_bin` is intentionally **not created** in this Feature Engineering notebook.

This feature requires **binning horsepower into quantile-based ranges** (Low / Medium / High / Extreme).  
Since quantiles depend on the **distribution of the training set**, computing them on the full dataset would introduce **data leakage**.

For this reason:

- the quantiles will be computed **only on the training split**
- the binning function will be applied to both train and test using **train-only thresholds**
- the feature will be created in the **preprocessing notebook**, after the train/test split

This ensures that no information from the test set influences the feature construction.

### 3.8 Logarithm Price

In [0]:
df["log_price"] = np.log1p(df["price_usd"])

### 3.9 Performance Composite Features

**Rationale for Performance Score**

This composite feature aggregates key performance-related attributes into a single score:  
- Power (horsepower)  
- Force (torque)  
- Speed (top speed)  
- Penalizes slower acceleration (0-100 km/h)

Scaling factors bring variables with different magnitudes onto a comparable range.  
The resulting feature provides a simplified representation of overall vehicle performance.

**Important**: This feature is created for **all models** (not just Linear Regression).  
While it helps reduce multicollinearity in linear models, tree-based models can also benefit from this high-level performance indicator.

In [0]:
df["performance_score"] = (
    df["horsepower_mean"] / 10 +
    df["torque_nm"] / 100 +
    df["top_speed_kmh"] / 100 -
    df["acc_0_100_mean"]
)

**Rationale for Acceleration Missing Flag**

This binary indicator captures **structural missingness** in acceleration data.  
Missingness is not random but concentrated in commercial vehicles (e.g., trucks, vans) where 0-100 km/h performance is not typically measured or relevant.

Instead of imputing or removing these values, this flag explicitly encodes the **absence of performance information**, allowing models to distinguish between:
- Vehicle categories where acceleration is a meaningful metric  
- Vehicle categories where acceleration data is structurally unavailable

**Important**: This feature is created for **all models**.  
Linear models will use it as a predictor, while tree-based models can use it to improve split decisions.

In [0]:
df["acc_missing_flag"] = df["acc_0_100_mean"].isna().astype(int)

### 3.10 Unified Engineered Dataset

**Philosophy: Unified Feature Engineering**

All models (Linear Regression, Random Forest, Decision Trees) start from the **same engineered feature set**.  
This ensures:

1. **Fair comparison** - Performance differences reflect model capabilities, not feature availability  
2. **Reproducibility** - All experiments use the same data foundation  
3. **Maintainability** - Feature updates happen in one place  
4. **Interpretability** - Comparisons are meaningful and scientifically valid

---

**What stays in the unified dataset:**

✅ All performance features (horsepower, acceleration, torque, top speed)  
✅ Engineered features (segment, brand tiers, performance_score, acc_missing_flag)  
✅ **Structural features** (engine_displacement_cc, battery_capacity_kwh)  
✅ Powertrain indicators (is_ev, is_hybrid, is_phev, is_commercial)  
✅ Target variables (price_usd, log_price)

**What gets removed:**

❌ Raw text columns (fuel_type → replaced by fuel_macro)  
❌ Redundant descriptions (engine_description → info captured in features)

---

**Model-specific adaptations** (e.g., dropping engine_cc/battery_kwh for Linear Regression, scaling) are deferred to the **preprocessing notebooks** (05_1 and 05_2).  
This separation ensures clean boundaries: feature engineering creates the data, preprocessing adapts it to specific model requirements.

In [0]:
# Create unified engineered dataset
df_engineered = df.copy()

# Remove ONLY raw/redundant columns
# All other features (including engine_cc, battery_kwh) are kept
df_engineered = df_engineered.drop(columns=[
    'fuel_type',           # Raw → replaced by fuel_macro
    'engine_description'   # Redundant text description
])

print(f"✅ Unified engineered dataset created")
print(f"Shape: {df_engineered.shape}")
print(f"\nFeatures: {df_engineered.shape[1]} columns")

✅ Unified engineered dataset created
Shape: (1209, 31)

Features: 31 columns


In [0]:
# Validate no duplicates
print(f"Duplicates: {df_engineered.duplicated().sum()}")

# Check for missing values
print(f"\nMissing values per column:")
print(df_engineered.isna().sum()[df_engineered.isna().sum() > 0])

Duplicates: 0

Missing values per column:
engine_displacement_cc     148
battery_capacity_kwh      1105
acc_0_100_min                5
acc_0_100_max                5
acc_0_100_mean               5
seats                        1
performance_score            5
dtype: int64


In [0]:
df_engineered.head()

,make,model,engine_displacement_cc,battery_capacity_kwh,acc_0_100_min,acc_0_100_max,acc_0_100_mean,acc_0_100_is_range,acc_0_100_is_instant,is_commercial,horsepower_min,horsepower_max,horsepower_mean,horsepower_is_range,top_speed_kmh,seats,torque_nm,price_usd,is_ev,is_hybrid,is_phev,fuel_macro,segment,is_luxury_brand,is_performance_luxury_brand,is_premium_brand,acceleration_class,hp_class,log_price,performance_score,acc_missing_flag
0,Porsche,911 Carrera S Cabriolet,2981.0,NaN,3.9,3.9,3.9,0,0,0,443.0,443.0,443.0,0,304.0,4.0,530,129900.0,0,0,0,Petrol,Sportscar,0,1,0,Fast,High,11.774528,48.74,0
1,Porsche,911 Carrera 4,2981.0,NaN,4.4,4.4,4.4,0,0,0,379.0,379.0,379.0,0,293.0,4.0,450,113300.0,0,0,0,Petrol,Sportscar,0,1,0,Fast,High,11.637803,40.93,0
2,Porsche,911 Carrera 4 GTS,2981.0,NaN,3.3,3.3,3.3,0,0,0,473.0,473.0,473.0,0,311.0,4.0,570,137000.0,0,0,0,Petrol,Sportscar,0,1,0,Fast,High,11.827744,52.81,0
3,Porsche,911 Targa 4 GTS,2981.0,NaN,3.5,3.5,3.5,0,0,0,473.0,473.0,473.0,0,307.0,4.0,570,157300.0,0,0,0,Petrol,Sportscar,0,1,0,Fast,High,11.965916,52.57,0
4,Porsche,911 Turbo Cabriolet,3745.0,NaN,2.9,2.9,2.9,0,0,0,572.0,572.0,572.0,0,320.0,4.0,750,184100.0,0,0,0,Petrol,Supercar,0,1,0,Extreme,High,12.123240,65.00,0


## 4. Export Unified Engineered Dataset

### 4.1 Export to Unity Catalog Delta Table

The unified engineered dataset is exported to a **single Delta table** in Unity Catalog:  
`workspace.caraiproject.engineered_cars_data`

Both preprocessing notebooks (05_1_Preprocessing_LinearRegression and 05_2_Preprocessing_DecisionTree) will load from this **same source table**, ensuring:

✅ Identical train/test splits (same random_state on same data)  
✅ Fair model comparison (same data foundation)  
✅ Reproducible experiments  
✅ Single source of truth for features

In [0]:
# Export unified engineered dataset to Delta format in Unity Catalog

# Define the target table
target_catalog = "workspace"
target_schema = "caraiproject"
target_table = "engineered_cars_data"
full_table_name = f"{target_catalog}.{target_schema}.{target_table}"

print(f"\nTarget table: {full_table_name}")

# Add export timestamp
df_engineered['fe_timestamp'] = datetime.now()

# Convert pandas DataFrame to Spark DataFrame for Delta write
spark_df = spark.createDataFrame(df_engineered)

# Write to Delta table (overwrite mode)
spark_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(full_table_name)

print(f"✅ Unified engineered dataset exported to {full_table_name}")
print(f"   Shape: {df_engineered.shape}")
print(f"   Timestamp: {df_engineered['fe_timestamp'].iloc[0]}")


Target table: workspace.caraiproject.engineered_cars_data
✅ Unified engineered dataset exported to workspace.caraiproject.engineered_cars_data
   Shape: (1209, 32)
   Timestamp: 2026-05-06 12:06:30.691508


### 4.2 Export to CSV for PyCaret (Local VSCode)

In [0]:
# Export unified engineered dataset to CSV for local PyCaret experiments
# Use config constant for export path
export_path = os.path.join(PROCESSED_DATA_PATH, "cars_cleaned.csv")
df_engineered.to_csv(export_path, index=False)

print(f"Unified dataset exported to {export_path}")
print("   Excluded: fe_timestamp, model (high cardinality)")

✅ Unified dataset exported to cars_cleaned.csv


In [0]:
df_engineered.head()

,make,model,engine_displacement_cc,battery_capacity_kwh,acc_0_100_min,acc_0_100_max,acc_0_100_mean,acc_0_100_is_range,acc_0_100_is_instant,is_commercial,horsepower_min,horsepower_max,horsepower_mean,horsepower_is_range,top_speed_kmh,seats,torque_nm,price_usd,is_ev,is_hybrid,is_phev,fuel_macro,segment,is_luxury_brand,is_performance_luxury_brand,is_premium_brand,acceleration_class,hp_class,log_price,performance_score,acc_missing_flag,fe_timestamp
0,Porsche,911 Carrera S Cabriolet,2981.0,NaN,3.9,3.9,3.9,0,0,0,443.0,443.0,443.0,0,304.0,4.0,530,129900.0,0,0,0,Petrol,Sportscar,0,1,0,Fast,High,11.774528,48.74,0,2026-05-06 12:06:30.691508
1,Porsche,911 Carrera 4,2981.0,NaN,4.4,4.4,4.4,0,0,0,379.0,379.0,379.0,0,293.0,4.0,450,113300.0,0,0,0,Petrol,Sportscar,0,1,0,Fast,High,11.637803,40.93,0,2026-05-06 12:06:30.691508
2,Porsche,911 Carrera 4 GTS,2981.0,NaN,3.3,3.3,3.3,0,0,0,473.0,473.0,473.0,0,311.0,4.0,570,137000.0,0,0,0,Petrol,Sportscar,0,1,0,Fast,High,11.827744,52.81,0,2026-05-06 12:06:30.691508
3,Porsche,911 Targa 4 GTS,2981.0,NaN,3.5,3.5,3.5,0,0,0,473.0,473.0,473.0,0,307.0,4.0,570,157300.0,0,0,0,Petrol,Sportscar,0,1,0,Fast,High,11.965916,52.57,0,2026-05-06 12:06:30.691508
4,Porsche,911 Turbo Cabriolet,3745.0,NaN,2.9,2.9,2.9,0,0,0,572.0,572.0,572.0,0,320.0,4.0,750,184100.0,0,0,0,Petrol,Supercar,0,1,0,Extreme,High,12.123240,65.00,0,2026-05-06 12:06:30.691508


In [0]:
df_engineered.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1209 entries, 0 to 1209
Data columns (total 32 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   make                         1209 non-null   object        
 1   model                        1209 non-null   object        
 2   engine_displacement_cc       1061 non-null   float64       
 3   battery_capacity_kwh         104 non-null    float64       
 4   acc_0_100_min                1204 non-null   float64       
 5   acc_0_100_max                1204 non-null   float64       
 6   acc_0_100_mean               1204 non-null   float64       
 7   acc_0_100_is_range           1209 non-null   int64         
 8   acc_0_100_is_instant         1209 non-null   int64         
 9   is_commercial                1209 non-null   int64         
 10  horsepower_min               1209 non-null   float64       
 11  horsepower_max               1209 non-null   flo

In [0]:
import os
os.listdir()

['03_EDA.ipynb',
 'binary_features_usd_dt.pkl',
 'usd_features_lr.pkl',
 'numerical_features_lr.pkl',
 'categorical_features_lr.pkl',
 'binary_features_dt.pkl',
 'metrics_databricks_df_rf_test.csv',
 'X_train_usd_df.pkl',
 'X_test_usd_df.pkl',
 'y_train_usd_df.pkl',
 'y_test_usd_df.pkl',
 'preprocess_usd_dt.pkl',
 'numerical_features_usd_dt.pkl',
 'categorical_features_usd_dt.pkl',
 '06_1_Training_LinearRegression.ipynb',
 'X_train_dt.pkl',
 'X_test_dt.pkl',
 'y_train_dt.pkl',
 'y_test_dt.pkl',
 'preprocess_dt.pkl',
 'numerical_features_dt.pkl',
 'categorical_features_dt.pkl',
 '04_Feature_Engineering.ipynb',
 '01_Data_Gathering.ipynb',
 '05_1_Preprocessing_LinearRegression.ipynb',
 'logs.log',
 '07_Pycaret.ipynb',
 'X_test_pycaret_log.pkl',
 'y_test_pycaret_log.pkl',
 'y_test_pycaret_usd.pkl',
 'X_test_pycaret_usd.pkl',
 'cars_cleaned.csv',
 'X_train_lr_usd.pkl',
 'X_test_lr_usd.pkl',
 'y_train_lr_usd.pkl',
 'y_test_lr_usd.pkl',
 'numerical_features_lr_usd.pkl',
 'categorical_features